In [1]:
# Kill worker

In [2]:
from distributed import Client, Security
import os, signal, time

# Get this address from your currently running client
SCHEDULER = "tls://aritra-2emandal-40cern-2ech.dask.cmsaf-prod.flatiron.hollandhpc.org:8786"

security = Security(
    require_encryption=True,
    tls_ca_file="/etc/cmsaf-secrets/ca.pem",
    tls_client_cert="/etc/cmsaf-secrets/hostcert.pem",
    tls_client_key="/etc/cmsaf-secrets/hostcert.pem",
)

client = Client(SCHEDULER, security=security, timeout="30s")
client


<Client: 'tls://192.168.27.84:8786' processes=5 threads=5, memory=27.01 GiB>

2026-04-23 12:24:18,587 - distributed.client - ERROR - Failed to reconnect to scheduler after 30.00 seconds, closing client


In [9]:
# List workers so you can copy the exact stuck worker address
workers = client.scheduler_info()["workers"]

for i, (addr, meta) in enumerate(workers.items()):
    print(
        i,
        addr,
        "status=", meta.get("status"),
        "nthreads=", meta.get("nthreads"),
        "memory_limit_GB=", round(meta.get("memory_limit", 0) / 1e9, 2),
    )


0 tls://129.93.182.101:42771 status= running nthreads= 1 memory_limit_GB= 5.8
1 tls://129.93.182.101:44617 status= running nthreads= 1 memory_limit_GB= 5.8
2 tls://129.93.182.107:37225 status= running nthreads= 1 memory_limit_GB= 5.8
3 tls://129.93.182.107:40583 status= running nthreads= 1 memory_limit_GB= 5.8
4 tls://129.93.182.110:35191 status= running nthreads= 1 memory_limit_GB= 5.8
5 tls://129.93.182.111:34881 status= running nthreads= 1 memory_limit_GB= 5.8
6 tls://129.93.182.116:41065 status= running nthreads= 1 memory_limit_GB= 5.8
7 tls://129.93.182.117:35411 status= running nthreads= 1 memory_limit_GB= 5.8
8 tls://129.93.182.118:36921 status= running nthreads= 1 memory_limit_GB= 5.8
9 tls://129.93.183.22:32903 status= running nthreads= 1 memory_limit_GB= 5.8
10 tls://129.93.183.22:39335 status= running nthreads= 1 memory_limit_GB= 5.8


In [8]:
# Replace this with the exact worker address, or a unique substring
WORKER_MATCH = "tls://129.93.182.101:40353"

matches = [w for w in client.scheduler_info()["workers"] if WORKER_MATCH in w]
print("Matched workers:")
for w in matches:
    print(" ", w)

assert len(matches) == 1, f"Expected exactly one match, got {len(matches)}"

stuck_worker = matches[0]

# First try graceful scheduler-side retirement
client.retire_workers(
    workers=[stuck_worker],
    close_workers=True,
    remove=True,
)
print("Requested retire for:", stuck_worker)


Matched workers:
  tls://129.93.182.101:40353
Requested retire for: tls://129.93.182.101:40353


In [5]:
# Force-kill the worker process if it is still registered
still_there = stuck_worker in client.scheduler_info()["workers"]

if still_there:
    print("Worker still present; sending SIGKILL to worker process:", stuck_worker)
    client.run(
        lambda: os.kill(os.getpid(), signal.SIGKILL),
        workers=[stuck_worker],
        wait=False,
    )
else:
    print("Worker already gone:", stuck_worker)


Worker already gone: tls://129.93.182.113:38635
